# 03 — Segmentation RFM & KMeans

## Ce qu'on va faire ici
On part de notre table de 4 518 clients et on les classe en **groupes** 
selon leur comportement. À la fin, chaque client aura :
- Un **segment marketing** (Champion, À risque, Perdu, etc.) → lisible business
- Un **cluster KMeans** (numéro de groupe) → découvert par l'algorithme

## Deux approches, deux usages
- **Partie A — RFM classique** : la méthode marketing standard, basée sur 
  des règles claires. Idéale pour communiquer avec une équipe CRM.
- **Partie B — KMeans** : l'approche data-driven, qui laisse les données 
  parler. Idéale pour découvrir des profils inattendus.

On compare les deux à la fin.

## Important : on n'utilise PAS la cible CHURN ici
La segmentation est **descriptive**, pas prédictive. On veut comprendre 
"qui sont mes clients", pas "qui va churner". La prédiction, ce sera 
l'étape 4.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Chargement
DATA_PROCESSED = Path("../data/processed")
df = pd.read_csv(DATA_PROCESSED / "customer_features.csv")

print(f"✅ Table chargée : {len(df):,} clients × {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")

✅ Table chargée : 4,518 clients × 13 colonnes

Colonnes : ['customer_id', 'churn', 'n_orders_future', 'recency', 'frequency', 'monetary', 'avg_basket', 'std_basket', 'n_distinct_products', 'tenure_days', 'avg_interpurchase_days', 'n_cancellations', 'cancellation_rate']


# PARTIE A — SEGMENTATION RFM RULES-BASED

## 2. Calcul des scores R, F, M (1 à 5)

On découpe chaque dimension RFM en **5 groupes égaux** (quintiles) :
- Pour la **Recency** : plus c'est récent, mieux c'est → un client avec 
  une faible Recency obtient le score 5
- Pour la **Frequency** : plus c'est élevé, mieux c'est → score 5 pour 
  les clients qui commandent le plus
- Pour le **Monetary** : idem, plus c'est élevé, mieux c'est → score 5 
  pour les gros dépensiers

Chaque client obtient donc 3 scores entre 1 et 5.
